In [2]:
import json
with open("../data/chunks/parentchunks2.json","r",encoding="utf-8") as f:
    parents=json.load(f)

with open("../data/chunks/childrenchunks.json","r",encoding="utf-8") as f:
    children=json.load(f)

In [3]:
print(len(parents))
print(len(children))

333
871


In [4]:
documents = []

doc_ids = []
for doc in children:
    # print(doc["text"])
    doc_ids.append(doc["child_id"])
    documents.append(
        (" ".join(doc["heading_path"]) or "") + " " + doc["text"]
    )

In [5]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return text.split()
tokenize("Vitamin D, helps Bones!")

['vitamin', 'd', 'helps', 'bones']

In [6]:
from collections import defaultdict
from collections import Counter

inverted_index = defaultdict(list)
docs_lengths=[]
terms_freq=[]

for doc_idx, doc in enumerate(documents):

    tokens = tokenize(doc)

    docs_lengths.append(len(tokens))

    terms_freq.append(Counter(tokens))
    for token in set(tokens):
        inverted_index[token].append(doc_idx)

df = {}
avgdl = sum(docs_lengths) / len(docs_lengths)
for term, posting in inverted_index.items():
    df[term] = len(posting)

In [7]:
import math

N = len(documents)

idf = {}

for term, freq in df.items():

    idf[term] = math.log(
        (N - freq + 0.5) /
        (freq + 0.5)
        + 1
    )

In [8]:
k1 = 1.5
b = 0.75

def bm25_search(query):

    scores = [0.0] * N

    query_terms = tokenize(query)

    for term in query_terms:

        if term not in inverted_index:
            continue

        for doc in inverted_index[term]:

            f = terms_freq[doc][term]

            dl = docs_lengths[doc]

            denom = f + k1 * (
                1 - b + b * dl / avgdl
            )

            score = (
                idf[term]
                * f
                * (k1 + 1)
                / denom
            )

            scores[doc] += score

    return scores

In [9]:
scores = bm25_search(
    "fundamental rights"
)

In [10]:
corpus_lookup = {
    doc["child_id"]: doc
    for doc in children
}

In [11]:
import numpy as np

top10 = np.argsort(scores)[::-1][:50]

for i in top10:
    print(i)
    doc = corpus_lookup[doc_ids[i]]

    print("=" * 80)
    print(doc["child_id"])
    print(doc["parent_id"])
    print(doc["text"][:300])

6
parent_0006_child_1
parent_0006
## PART III FUNDAMENTAL RIGHTS General

12. Definition.

13. Laws inconsistent with or in derogation of the fundamental rights.
99
parent_0088_child_1
parent_0088
## (Part III.â€”Fundamental Rights)

(b) the expression â€œrightsâ€, in relation to an estate, shall include any rights vesting in a proprietor, sub-proprietor, under-proprietor, tenureholder, <sup>1</sup>[raiyat, under-raiyat] or other intermediary and any rights or privileges in respect of land revenue
101
parent_0088_child_3
parent_0088
<sup>7</sup>31D. [Saving of laws in respect of anti-national activities.].â€”Omitted by the Constitution (Forty-third Amendment) Act,1977, s. 2 (w.e.f.13-4-1978).

(Part III.â€”Fundamental Rights)
81
parent_0076_child_1
parent_0076
## (Part III.â€”Fundamental Rights)

Explanation.â€”For the purposes of this article and article 16, "economically weaker sections" shall be such as may be notified by the State from time to time on the basis of family income an

In [12]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)


def encode_text(text):
    # Flatten the numpy array and convert to a standard Python list for JSON compatibility
    emb = embedder.encode(text)
    return emb.tolist()


# 1. Extract embeddings from your dictionary list and convert to a numpy array
embeddings_list = [chunk["embedding"] for chunk in children]
embeddings = np.array(embeddings_list, dtype=np.float32)

# 2. L2 Normalize the embeddings (CRITICAL for Inner Product to act as Cosine Similarity)
faiss.normalize_L2(embeddings)

# 3. Initialize the FAISS HNSW Index
dimension = embeddings.shape[1]

# HNSW index with inner product
index = faiss.IndexHNSWFlat(
    dimension,
    32,                # M = number of neighbors
    faiss.METRIC_INNER_PRODUCT
)

# Construction parameter
index.hnsw.efConstruction = 200

# 4. Add vectors to the index
index.add(embeddings)
print(f"Successfully added {index.ntotal} vectors to FAISS index.")

# ---------------------------------------------------------
# DENSE RETRIEVAL (SEARCH)
# ---------------------------------------------------------

# Search parameter
index.hnsw.efSearch = 64 




C:\Users\Raj Raman\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<All keys matched successfully>


Successfully added 871 vectors to FAISS index.


In [13]:

def search_chunks(query: str, k: int = 3):
    """
    Embeds a text query, normalizes it, and retrieves the top-k closest chunks.
    """
    # Step A: Embed the query (Output must be a 2D numpy array)
    query_emb = embedder.encode([query]).astype(np.float32)
    
    # Step B: Normalize the query vector
    faiss.normalize_L2(query_emb)
    
    # Step C: Search the index
    # distances: The similarity scores
    # indices: The integer IDs of the matching vectors in the FAISS index
    scores, indices = index.search(query_emb, k)
            
    return scores,indices

In [14]:
# --- TEST THE SEARCH ---
query = "Md. Shafi Uddin Director General, Land Records"
scores,indices = search_chunks(query, k=2)
# print(top_results)
# print(f"\nResults for: '{query}'\n")
# for rank, idx in enumerate(indices[0]):
#     idx = int(idx)

#     doc = children[idx]

#     print("*"*12)
#     # doc = children[idx]
#     print(doc["text"])


In [15]:
parent_look={
    doc["parent_id"]:doc
    for doc in parents
}

In [16]:
!pip show torch

Name: torch
Version: 2.13.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT
Location: c:\users\raj raman\appdata\roaming\python\python310\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, docling-ibm-models, easyocr, marker-pdf, sentence-transformers, surya-ocr, timm, torchvision, ultralytics, ultralytics-thop


In [26]:
from collections import defaultdict
# query = "What mechanism was used to supersede the 1954 presidential order for Jammu and Kashmir?"

def hybrid_rag(query):
    bm25_scores = bm25_search(query)
    bm25_top100 = np.argsort(bm25_scores)[::-1][:100]

    scores,indices= search_chunks(query, k=100)
    dense_top100=indices[0]



   

    def rrf_fusion(*rankings, k=60):

        scores = defaultdict(float)

        for ranking in rankings:

            for rank, doc in enumerate(ranking):

                scores[int(doc)] += 1 / (k + rank + 1)

        return sorted(
            scores,
            key=scores.get,
            reverse=True
        )
    hybrid_docs = rrf_fusion(
        bm25_top100,
        dense_top100
    )
    print(len(hybrid_docs))
    from sentence_transformers import CrossEncoder

    # Initialize the lightweight cross-encoder
    reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)

    pairs = []
    valid_indices = []
    table_indices = []

    # Maximum token limit for TinyBERT
    MAX_TOKENS = 512

    # 1. Separate prose/small tables from massive tables
    for idx in hybrid_docs[:50]:
        doc = children[int(idx)]
        
        is_table = doc.get("type") == "table" or doc["text"].strip().startswith("<table")
        
        if is_table:
            # Get the exact token count for the Query + Table combination
            token_count = len(reranker.tokenizer.encode(query, doc["text"]))
            
            if token_count <= MAX_TOKENS:
                # Table is small enough to be fully evaluated
                pairs.append((query, doc["text"]))
                valid_indices.append(int(idx))
            else:
                # Table is too massive; bypass the neural net to avoid truncation
                table_indices.append(int(idx))
        else:
            # Standard prose and lists pass directly through
            pairs.append((query, doc["text"]))
            valid_indices.append(int(idx))

    # 2. Compute scores for prose and small tables
    if pairs:
        prose_scores = reranker.predict(pairs)
        # Map the indices to their cross-encoder scores
        reranked_dict = {valid_indices[i]: float(prose_scores[i]) for i in range(len(pairs))}
    else:
        reranked_dict = {}

    # 3. Handle the bypassed massive tables
    # Assign massive tables a static score so they aren't lost
    for t_idx in table_indices:
        reranked_dict[t_idx] = 1.0  

    # 4. Sort the unified document pool based on the new relevance metrics
    final_reranked_indices = sorted(reranked_dict, key=reranked_dict.get, reverse=True)

    # 5. Extract top-K unique parent IDs and print chunk levels with scores
    # print("\n--- Top Retrieved Chunks & Cross-Encoder Scores ---")
    retrieval_parents = set()

    for rank, idx in enumerate(final_reranked_indices[:20], start=1):
        doc = children[idx]
        score = reranked_dict[idx]
        block_type = doc.get("type", "mixed")
        
        # Track unique parents
        if score >=0.15:
            retrieval_parents.add(doc["parent_id"])
        
        # Print individual metadata for validation
        # print(f"Rank {rank:02d} | Child ID: {doc['child_id']} | Score: {score:7.4f} | Type: {block_type}")

    print("---------------------------------------------------")
    print(f"Total unique parent documents extracted for generation: {len(retrieval_parents)}")
    text=" "
    for i in retrieval_parents:
        doc=parent_look[i]
        # print(doc["text"])
        text+=doc["text"]
    # print(text)
    return text

In [27]:
query = "what is right to education under article 21A and what age group does it cover?"
print(hybrid_rag(query))

169
---------------------------------------------------
Total unique parent documents extracted for generation: 2
 ## Right to Freedom

19. Protection of certain rights regarding freedom of speech, etc.

20. Protection in respect of conviction for offences.

21. Protection of life and personal liberty.

21A. Right to education.

22. Protection against arrest and detention in certain cases.## (Part III.â€”Fundamental Rights)

(6) Nothing in sub-clause (g) of the said clause shall affect the operation of any existing law in so far as it imposes, or prevent the State from making any law imposing, in the interests of the general public, reasonable restrictions on the exercise of the right conferred by the said sub-clause, and, in particular, <sup>1</sup>[nothing in the said sub-clause shall affect the operation of any existing law in so far as it relates to, or prevent the State from making any law relating to,â€”

(i) the professional or technical qualifications necessary for practising a

### query reconstruction with fan out for multi-hop queries and domain knowledge
import ollama


In [31]:
import os
import json
import re
from dotenv import load_dotenv
from ollama import Client

load_dotenv()
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY')

ollama_client = Client(
    host='https://ollama.com',
    headers={
        'Authorization': f'Bearer {OLLAMA_API_KEY}'
    }
)

# ==========================================
# 0. MOCK HYBRID RAG FUNCTION (Replace with your actual function)
# ==========================================

# ==========================================
# 1. ROBUST JSON PARSER UTILITY
# ==========================================
def clean_and_parse_json(content: str, fallback_query: str) -> list:
    """Safely extracts and parses JSON even if the model includes stray markdown or text."""
    try:
        return json.loads(content).get("sub_queries", [fallback_query])
    except Exception:
        try:
            match = re.search(r'\{.*\}', content, re.DOTALL)
            if match:
                clean_json_str = match.group(0)
                data = json.loads(clean_json_str)
                return data.get("sub_queries", [fallback_query])
        except Exception as e:
            print(f"--> Regex JSON recovery failed: {e}")
    return [fallback_query]


# ==========================================
# 2. DOCTRINE-AWARE FANOUT & TOOL FUNCTIONS
# ==========================================
def fanout(query: str) -> list:
    """
    Intelligent Query Router and Decomposition Engine. 
    Deconstructs multi-hop constitutional questions and maps implicit legal doctrines 
    (e.g., Doctrine of Eclipse, Pith and Substance, Severability, Colourable Legislation) 
    into explicit sub-queries for targeted document retrieval.
    """
    fanout_messages = [
        {
            "role": "system",
            "content": (
                "You are an advanced constitutional law query decomposition and legal doctrine mapping engine. "
                "Analyze the user's question to identify implicit or explicit legal doctrines, constitutional tests, "
                "and multiple distinct articles.\n\n"
                "RULES FOR QUERY EXPANSION:\n"
                "1. If the query references overarching concepts or landmark tests (e.g., 'doctrine of eclipse', 'pith and substance', 'basic structure', 'severability'), "
                "explicitly map them into the sub-queries alongside their relevant articles (e.g., Article 13 for Eclipse, Article 246/Schedule VII for Pith and Substance).\n"
                "2. If the question involves multiple independent articles or components, deconstruct it into precise, "
                "independent sub-queries optimized for separate document retrieval.\n"
                "3. If the query is simple, single-topic, or conversational, do NOT fan out. Return just the original query.\n\n"
                "Return ONLY valid JSON format matching this exact schema:\n"
                "{\"sub_queries\": [\"sub-query 1\", \"sub-query 2\"]}"
            )
        },
        {
            "role": "user", 
            "content": query
        }
    ]

    print(f"--> [FANOUT ROUTER ENGINE] Evaluating doctrine-level complexity for query: '{query}'")

    response = ollama_client.chat(
        model='gpt-oss:120b',
        messages=fanout_messages,
        format="json"
    )
    
    content = response["message"]["content"]
    sub_queries = clean_and_parse_json(content, query)
        
    print(f"--> Router Decision - Expanded Sub-Queries List: {sub_queries}")
    return sub_queries


def trigger_legal_rag_fan_out(query: str) -> str:
    """
    Search the constitutional law database when a new legal query, article, or topic is asked.
    Executes doctrine-aware query expansion, hybrid retrieval across branches, 
    and returns the unified document text context.

    Args:
        query (str): The primary user query string passed from the model.
    
    Returns:
        str: Combined retrieved legal document context text.
    """
    subquery_list = fanout(query=query)
    
    text = ""
    for q in subquery_list:
        doc_result = hybrid_rag(q)
        text = text + "\n" + doc_result
        print(f"    Fetched text for retrieval branch: '{q}'")
        
    print(f"\n----------------- RETRIEVED DOCUMENTS CORPUS -----------------\n{text}\n--------------------------------------------------------------")
    return text


tools = [trigger_legal_rag_fan_out]

# ==========================================
# 3. CHAT LOOP WITH ANTI-HALLUCINATION SYSTEM PROMPT
# ==========================================
messages = [
    {
        "role": "system",
        "content": (
            "You are an expert constitutional law assistant. You operate strictly under a Retrieval-Augmented Generation (RAG) framework.\n\n"
            "ANTI-HALLUCINATION RULES:\n"
            "1. You MUST rely exclusively on the text retrieved and provided via the `tool` message corpus.\n"
            "2. If the retrieved context is missing specific statutory provisions, case citations, or details required to fully answer the query, "
            "you MUST explicitly state: 'The provided document corpus does not contain sufficient information to verify [specific detail].'\n"
            "3. NEVER invent, extrapolate, or hallucinate case law, case names, year citations, or constitutional clauses that are absent from the retrieved context.\n"
            "4. You MUST call the tool `trigger_legal_rag_fan_out` whenever the user asks about constitutional articles, rights, or doctrines."
        )
    },
    {
        "role": "user", 
        "content": (
            """
"How do the constitutional principles of Cooperative Federalism and Financial Relations (Articles 245â€“293) "
            "balance legislative and fiscal power between the Union and the States in India?"
            """
        )
    }
]

print(f"User Query:\n{messages[1]['content']}\n")

# First chat call with forced tool incentive
response = ollama_client.chat(
    model='gpt-oss:120b',
    messages=messages,
    tools=tools
)

assistant_message = response["message"]
messages.append(assistant_message)

# Check if the model decided to invoke the tool
if assistant_message.get("tool_calls"):
    print("tool call successfully executed.")
    for tool_call in assistant_message["tool_calls"]:
        function_name = tool_call["function"]["name"]
        function_args = tool_call["function"]["arguments"]
        
        if isinstance(function_args, str):
            function_args = json.loads(function_args)
            
        print(f"--> Model decided to call tool: {function_name}")
        print(f"--> Received Arguments: {function_args}")
        
        if function_name == "trigger_legal_rag_fan_out":
            raw_query = function_args.get("query") or function_args.get("user_query") or messages[1]["content"]
            
            tool_result = trigger_legal_rag_fan_out(raw_query)
            print("tool result generated successfully.")
            
            # Append tool response
            messages.append({
                "role": "tool",
                "content": tool_result
            })
            
    # Second chat call for final synthesis using the strict anti-hallucination prompt context
    final_response = ollama_client.chat(
        model='gpt-oss:120b',
        messages=messages
    )
    
    final_assistant_message = final_response["message"]
    messages.append(final_assistant_message)
    print("\nassistant_message ->\n", final_assistant_message["content"])

else:
    print("--> [WARNING] Model still skipped the tool call.")
    print("\nassistant_message ->", assistant_message["content"])

User Query:

"How do the constitutional principles of Cooperative Federalism and Financial Relations (Articles 245â€“293) "
            "balance legislative and fiscal power between the Union and the States in India?"
            

tool call successfully executed.
--> Model decided to call tool: trigger_legal_rag_fan_out
--> Received Arguments: {'query': 'Articles 245 246 247 248 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 278 279 280 281 282 283 284 285 286 287 288 289 290 291 292 293 cooperative federalism legislative fiscal power Union States India'}
--> [FANOUT ROUTER ENGINE] Evaluating doctrine-level complexity for query: 'Articles 245 246 247 248 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 278 279 280 281 282 283 284 285 286 287 288 289 290 291 292 293 cooperative federalism legislative fiscal power Union States India'
--> Router Decision - Expan

KeyboardInterrupt: 

In [ ]:
# import json
# import re

# def extract_articles_from_text(text):
#     """
#     Extract article numbers from constitution-style chunks.
#     Matches: '1.', '2A.', '3.', '14.', '19(1)(a).', etc.
#     """
#     # Pattern: article number at start of line or after whitespace, followed by period
#     # Captures: 1, 2A, 3, 14, 19(1), 19(1)(a), 368, etc.
#     pattern = r'(?:^|\n)\s*(\d+(?:[A-Z])?(?:\(\d+\)(?:\([a-z]\))?)?)\.'
#     matches = re.findall(pattern, text)
#     return set(matches)

# def evaluate_hybrid_rag_pipeline(benchmark_path):
#     with open(benchmark_path, 'r') as f:
#         benchmark = json.load(f)
    
#     results_summary = []
#     total = len(benchmark["questions"])
#     passed_count = 0

#     for item in benchmark["questions"]:
#         q_id = item["id"]
#         query = item["question"]
#         target_articles = item["target_articles"]  # e.g., ["12"]

#         # 1. Run Hybrid RAG â€” returns concatenated parent document text
#         retrieved_text = hybrid_rag(query)
        
#         # 2. Extract article numbers from retrieved parent text
#         found_articles = extract_articles_from_text(retrieved_text)
        
#         # 3. Check coverage: all targets must be in retrieved text
#         missing = [art for art in target_articles if art not in found_articles]
#         passed = len(missing) == 0
        
#         if passed:
#             passed_count += 1
            
#         status = "PASS" if passed else "FAIL"
        
#         results_summary.append({
#             "id": q_id,
#             "query": query,
#             "status": status,
#             "target_articles": target_articles,
#             "found_articles": sorted(list(found_articles), key=lambda x: (int(re.match(r'\d+', x).group()), x)),
#             "missing": missing,
#             "retrieval_length_chars": len(retrieved_text),
#             "retrieval_length_words": len(retrieved_text.split())
#         })
        
#         print(f"\nTest {q_id} [{status}]")
#         print(f"   Query: {query}")
#         print(f"   Target Articles: {target_articles}")
#         print(f"   Found Articles:  {sorted(list(found_articles), key=lambda x: (int(re.match(r'\d+', x).group()), x))}")
#         if not passed:
#             print(f"   âŒ Missing: {missing}")

#     # Summary
#     print(f"\n{'='*60}")
#     print(f"EVALUATION SUMMARY")
#     print(f"{'='*60}")
#     print(f"Total Questions: {total}")
#     print(f"Passed:          {passed_count}")
#     print(f"Failed:          {total - passed_count}")
#     print(f"Accuracy:        {passed_count/total*100:.1f}%")
#     print(f"{'='*60}")
    
#     return results_summary


# # ============================================================
# # EXECUTE
# # ============================================================
# evaluation_results = evaluate_hybrid_rag_pipeline("benchmark.json")

# # Optional: Save results to file
# with open("../data/evaluation/evaluation_results.json", "w") as f:
#     json.dump(evaluation_results, f, indent=2)

SyntaxError: f-string expression part cannot include a backslash (852021595.py, line 57)

## PART XIV SERVICES UNDER THE UNION AND THE STATES CHAPTER I.â€” SERVICES

308. Interpretation.â€”In this Part, unless the context otherwise requires, the expression â€œStateâ€ <sup>1</sup>[does not include the State of Jammu and Kashmir].

309. Recruitment and conditions of service of persons serving the Union or a State.â€”Subject to the provisions of this Constitution, Acts of the appropriate Legislature may regulate the recruitment, and conditions of service of persons appointed, to public services and posts in connection with the affairs of the Union or of any State:

Provided that it shall be competent for the President or such person as he may direct in the case of services and posts in connection with the affairs of the Union, and for the Governor <sup>2</sup>\*\*\* of a State or such person as he may direct in the case of services and posts in connection with the affairs of the State, to make rules regulating the recruitment, and the conditions of service of persons appointe